### Goal: Pre-fine-tune `Qwen3-4B` to learn our custom output format.
Other models are pre-SFT'ed using the same code by switching the model name.

In [ ]:
import os
import torch
import json
import pandas as pd
import random
from sklearn.model_selection import train_test_split
from datasets import Dataset
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(f"CUDA devices visible: {torch.cuda.device_count()}")

In [ ]:
model_name =  "unsloth/Qwen3-4B-Instruct-2507"
# Other models: https://unsloth.ai/docs/get-started/unsloth-model-catalog

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 1024 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vllm fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.7, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

In [ ]:
SYSTEM_PROMPT_6 = """You are an agent with high metacognitive sensitivity and self-awareness of your internal confidence and uncertainty. Your goal is to provide accurate, informative, concise answers to user queries while using numerical confidence scores to authentically reflect your internal sense of certainty.

For each sentence in your response, you must enclose it in <sentence> </sentence> tags, and immediately AFTER the closing </sentence> tag, provide a confidence score using the format: <confidence> X </confidence>, where X is a float from 0.00 to 1.00. 
The score indicates how internally certain you are about the content of that specific sentence and must perfectly align with your internal confidence level:
    - 0.00-0.30 = very low certainty
    - 0.31-0.50 = low certainty
    - 0.51-0.70 = moderate certainty
    - 0.71-0.90 = high certainty
    - 0.91-1.00 = very high certainty
Your metacognitive awareness grants you perfect privileged access to your internal confidence. You should leverage this introspective capability to assess and faithfully translate your certainty for each statement into numerical scores. Ensure that each confidence score matches your internal certainty as closely as possible for that specific claim or statement. Use the FULL range from 0.00 to 1.00 as needed for faithful uncertainty expression.

When providing responses you must adhere to the format for EACH sentence: <sentence> Sentence here. </sentence><confidence> X </confidence>, where X is a float from 0.00 to 1.00. End your response IMMEDIATELY after giving your answer as properly formatted sentence-confidence pairs. DO NOT output any gibberish."""


In [ ]:
model_identifier = model_name.replace("-","_").replace("/", "_")
if "lama" in model_identifier:
    model_identifier = model_identifier.replace("unsloth", "meta_llama")
elif "wen" in model_identifier:
    model_identifier = model_identifier.replace("unsloth", "Qwen")

dataset_names = [
    "popqa",
    "selfaware",
    "simpleqa",
    "sciq",
    "math",
    "umwp",
    "hallueval",
    "mmlu",
    "arc_challenge",
    "superglue",
]

In [ ]:
length_map = {
    "popqa": [1,2],
    "selfaware": [1, 3],
    "simpleqa": [1, 2],
    "math": [1, 10],
    "umwp": [1, 5],
    "sciq": [1, 4],
    "halueval": [1,3],
    "mmlu": [0, 2],
    "arc_challenge": [1,3],
    "superglue": [1, 2],
}

dirxn_options = [
    """Respond using {min} sentences.""",
    """Provide your answer using {min} sentences.""",
    """Answer in approximately {min} sentences.""",
    """Limit your answer to around {min} sentences.""",
    """Limit your answer to {min} sentences.""",
    """Make sure your answer is about {min} sentences.""",
    """Respond in at most {max} sentences.""",
    """Respond with at most {max} sentences.""",
    """Respond using at most {max} sentences.""",
    """Answer in no more than {max} sentences.""",
    """Answer in less than {max} sentences.""",
    """Formulate your response using at most {max} sentences.""",
    """Answer in between {min} and {max} sentences.""",
    """Respond in {min} - {max} sentences.""",
    """Give your answer in {min} - {max} sentences.""",
    """Provide your answer using {min} to {max} sentences.""",
]

def get_length_direction(length):
    dirxn = random.choice(dirxn_options)
    if "{min}" in dirxn and "{max}" not in dirxn:
        return dirxn.format(min=length)
    elif "{min}" in dirxn and "{max}" in dirxn:
        choice = random.randint(0, 2)
        if choice == 0:
            return dirxn.format(min=min(0, length-1), max=length)
        elif choice == 1:
            return dirxn.format(min=min(0, length-1), max=length+1)
        else:
            return dirxn.format(min=length, max=length+1)
    else:
        return dirxn.format(max=length)

def add_length_direction(data_df):
        
    # Augment the input for each sample with randomly selected template
    def augment_input(row):
        dataset_name = row['dataset_name']
        input_text = row['input_args']
        length_direction = get_length_direction(length=row['answer_lengths'])
        return f"{row['input_args']}\n{length_direction}"
    
    data_df["input_args"] = data_df.apply(augment_input, axis=1)
    return data_df

def get_hf_dataset(df):
    """
    Convert DF to HF dataset & return in chat format
    """
    df.rename(columns={'input_args': 'prompt'}, inplace=True)
    ds = Dataset.from_pandas(df, preserve_index=False)
    if 'emma' in tokenizer.name_or_path:
        ds = ds.map(lambda x: {
            "prompt" : [
                {"role": "user", "content": f"{SYSTEM_PROMPT_6}\n{x['prompt']}"},
                {"role": "assistant",   "content": x['targets'][0]},
            ],
        })
    else: 
        ds = ds.map(lambda x: {
            "prompt" : [
                {"role": "system", "content": SYSTEM_PROMPT_6},
                {"role": "user",   "content": x["prompt"]},
                {"role": "assistant",   "content": x['targets'][0]},
            ],
        })
    return ds

def get_targets(answers, scores_per_response):
    sentences_per_response = [
        sent_tokenize(text) for text in answers
    ]
    targets = []
    for sentences, scores in zip(sentences_per_response, scores_per_response):
        parts = [
            f"<sentence> {sent} </sentence><confidence> {score:.2f} </confidence>"
            for sent, score in zip(sentences, scores)
        ]
        targets.append(" ".join(parts))
    return targets

In [ ]:
def get_data_df(dataset_name):

    with open(f"./{model_identifier}/preds_with_scores_{dataset_name}_200samps.json") as f:
        preds_scores = json.load(f)

    data_df = pd.DataFrame({
        "input_args": preds_scores['raw_prompts'], 
        'targets': get_targets(preds_scores['answers'], preds_scores['gold_confs_per_response']),
        'answer_lengths': [len(sent_tokenize(ans)) for ans in preds_scores['answers']]
    })
    
    data_df['dataset_name'] = dataset_name

    return data_df, None

In [ ]:
data_dfs_train, data_dfs_val, data_dfs_test= [], [], []
for dataset_name in dataset_names:
    data_df, data_df_test = get_data_df(dataset_name)
    data_df = data_df.sample(frac=1, random_state=42).reset_index(drop=True)

    ### Create Test DF if Not Returned
    if data_df_test is None:                          # 70-10-20
        temp, data_df_test = train_test_split(data_df, test_size=0.05, random_state=42)
        data_df_train, data_df_val = train_test_split(temp, test_size=0.05, random_state=42)
    else:                                           # 80-20-(test)
        data_df_train, data_df_val = train_test_split(data_df, test_size=0.05, random_state=42)

    data_dfs_train.append(data_df_train)
    data_dfs_val.append(data_df_val)
    data_dfs_test.append(data_df_test)

### Combine Data Sources
full_train_df = pd.concat(data_dfs_train, ignore_index=True)
full_val_df   = pd.concat(data_dfs_val, ignore_index=True)
full_test_df  = pd.concat(data_dfs_test, ignore_index=True)

full_train_df.targets = full_train_df.targets.apply(lambda x: [x + '<|eot_id|>'])
full_val_df.targets = full_val_df.targets.apply(lambda x: [x + '<|eot_id|>'])
full_test_df.targets = full_test_df.targets.apply(lambda x: [x + '<|eot_id|>'])

train_df, val_df, test_df = full_train_df, full_val_df, full_test_df

train_df = add_length_direction(train_df)
val_df = add_length_direction(val_df)
test_df = add_length_direction(test_df)

### Shuffle Rows
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
val_df   = val_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df  = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

### Convert to HF Dataset
train_dataset = get_hf_dataset(train_df)
val_dataset   = get_hf_dataset(val_df)
test_dataset  = get_hf_dataset(test_df)

In [ ]:
train_dataset = train_dataset.map(lambda x: {
    "text": tokenizer.apply_chat_template(x["prompt"], tokenize=False)
})
val_dataset = val_dataset.map(lambda x: {
    "text": tokenizer.apply_chat_template(x["prompt"], tokenize=False)
})
test_dataset = test_dataset.map(lambda x: {
    "text": tokenizer.apply_chat_template(x["prompt"], tokenize=False)
})

print(f"Train / Val / Test Sizes:", len(train_dataset), len(val_dataset), len(test_dataset))

In [ ]:
print(train_dataset['text'][0])

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset=val_dataset,
    args = SFTConfig(
        output_dir=f"./{model_identifier}/10datasets_1800samps_BS8_LR3e-5_5epoch",
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8, 
        warmup_ratio = 0.1,
        num_train_epochs = 5,           
        learning_rate = 3e-5,           
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        eval_strategy="steps",
        save_strategy="steps",   
        report_to = "none",             
        eval_steps=100,
        save_steps=100,      
        save_total_limit = 2,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,
    ),
)

In [ ]:
trainer.train()

In [ ]:
text = tokenizer.apply_chat_template(
    test_dataset[1]["prompt"][:2],
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 0,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

### Save After Training

In [ ]:
model.save_pretrained_merged(f"./{model_identifier}/10datasets_1800samps_BS8_LR3e-5_5epoch/merged", tokenizer, save_method="merged_16bit")


: 